***

# **About Automation**

***

This file is focused on automating the About pages on every excel file that we make for the indicators. The user needs to define the `sample_type`, `indicator_name`, dataframe that the possible years will be pulled from, and the date will be pulled automatically. The code also has the process of converting the `About Indicators.xlsx` file to a YAML type. From the YAML file, `dict_about.yaml`, more edits were made to include additional indicators. From this file, users can indicate the about section of a page by searching the file for the indicator name, and then editing the section dedicated to this indicator. 

The code works by linking to this YAML file, and extracting the information of the specific sample and indicator name the user wants. From there, this subset is converted to a pandas dataframe. This dataframe is parsed through to check for dynamic data (fields that change values regularly), such as year(s) or when the indicator was last updated. Afterwards, the notes are checked and cleaned for visual clarity. The data is returned as a dataframe, this way the user can see what the about page will look like for themselves. 

***

Preparing Workspace

***

In [ ]:
import pandas as pd
import os
import yaml
from datetime import date
import numpy as np
from tqdm import tqdm

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'Census')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_code    = os.path.join(path_git, 'Python Code', 'Census')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')
path_yaml    = os.path.join(path_config0, 'about_indicators.yaml')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

About Documentation

***

In [ ]:

try:
    with open(path_yaml, 'r') as yaml_file:
        dict_about = yaml.load(yaml_file, Loader=yaml.SafeLoader)
    print(dict_about)
except FileNotFoundError:
    print(f"Error: The file at {path_yaml} does not exist.")
except Exception as e:
    print(f"An error occurred: {e}")

dict_about

In [ ]:
estimates = list(dict_about.keys())
estimates.remove('ACS1')

list_samples = []
for estimate in estimates:
    if estimate in ['ACS5', 'BLS']:
        list_samples.extend(dict_about[estimate].keys())
samples = unique(list_samples)


list_indicators = []
for estimate in estimates:
    if estimate in ['ACS5', 'BLS']:
        for sample in samples:
            try:
                list_indicators.extend(dict_about[estimate][sample].keys())
            except: pass
    else:
        list_indicators.extend(dict_about[estimate].keys())
indicators = unique(list_indicators)

print(estimates)
print(samples)
print(indicators)

In [ ]:
list_df_about = []

for estimate in estimates:
    if estimate in ['ACS5', 'BLS']:
        for sample in samples:
            for indicator in indicators:
                try:
                    df_about = pd.DataFrame.from_dict(dict_about[estimate][sample][indicator]).T.reset_index().rename(columns = {'index': 'Indicator', 0: indicator})
                    df_about.loc[df_about['Indicator'] == 'Last Updated', indicator] = date.today().strftime('%Y-%m-%d')
                    
                    notes_row = df_about[df_about['Indicator'] == 'Notes'].copy()
                    notes = notes_row[indicator].values[0]
                    lines = notes.split('\\n')
                    new_rows = [{'Indicator': 'Notes' if i == 0 else '', indicator: line} for i, line in enumerate(lines) if line]
                    new_df = pd.DataFrame(new_rows)
                    df_filtered = df_about[df_about['Indicator'] != 'Notes']
                    df_about = pd.concat([df_filtered, new_df], ignore_index=True)
                    
                    list_df_about.append(df_about)
                    
                except: pass
    else:
        for indicator in indicators:
                try:
                    df_about = pd.DataFrame.from_dict(dict_about[estimate][indicator]).T.reset_index().rename(columns = {'index': 'Indicator', 0: indicator})
                    df_about.loc[df_about['Indicator'] == 'Last Updated', indicator] = date.today().strftime('%Y-%m-%d')
                    
                    notes_row = df_about[df_about['Indicator'] == 'Notes'].copy()
                    notes = notes_row[indicator].values[0]
                    lines = notes.split('\\n')
                    new_rows = [{'Indicator': 'Notes' if i == 0 else '', indicator: line} for i, line in enumerate(lines) if line]
                    new_df = pd.DataFrame(new_rows)
                    df_filtered = df_about[df_about['Indicator'] != 'Notes']
                    df_about = pd.concat([df_filtered, new_df], ignore_index=True)
                    
                    list_df_about.append(df_about)
                    
                except: pass
        
display(list_df_about[0], list_df_about[30])

***

Exporting

***

In [ ]:
print('')
print('Replacing current documentation workbook...')

for df in list_df_about:
    with pd.ExcelWriter(os.path.join(path_main, 'About Indicators.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace') as writer:
        df.to_excel(writer, index = False, sheet_name = df.columns[1], header = False)

print('Success')